In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import anthropic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data/structured/daily_sales.csv"
TEXT_PATH = PROJECT_ROOT / "data/unstructured/"

# Load data
df = pd.read_csv(CSV_PATH)
df['date'] = pd.to_datetime(df['date'])

# Load text files
text_files = {f.stem: f.read_text() for f in TEXT_PATH.glob("*.txt")}

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✅ CSV loaded: {len(df)} rows")
print(f"✅ Text files loaded: {len(text_files)}")
print(f"✅ Embedding model ready")
print(f"✅ Anthropic client ready")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1734.26it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CSV loaded: 1000 rows
✅ Text files loaded: 10
✅ Embedding model ready
✅ Anthropic client ready


In [3]:
def classify_query(question: str) -> str:
    """Route question to csv, text, or hybrid source."""
    q = question.lower()
    
    # Check hybrid first — questions that need both sources
    hybrid_keywords = ["best", "recommend", "highly rated", "sells well", 
                       "popular", "top rated", "worth buying"]
    csv_keywords = ["revenue", "sales", "region", "volume", "units", 
                    "december", "october", "november", "total", "highest",
                    "lowest", "how much", "how many"]
    text_keywords = ["feature", "customers say", "review", "details", "describe", 
                     "specification", "what is", "cleaning", "quality",
                     "key features", "what do"]
    
    # Hybrid: has both analytical and review intent
    if any(w in q for w in hybrid_keywords):
        return "hybrid"
    
    # CSV: purely analytical
    if any(w in q for w in csv_keywords):
        return "csv"
    
    # Text: purely descriptive
    if any(w in q for w in text_keywords):
        return "text"
    
    # Default to hybrid
    return "hybrid"

# Test on all 6 questions
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?"
]

print("=== Query Router Test ===\n")
for q in test_questions:
    qtype = classify_query(q)
    print(f"Q: {q[:65]}")
    print(f"→ {qtype}\n")

=== Query Router Test ===

Q: What was the total revenue for Electronics category in December 2
→ csv

Q: Which region had the highest sales volume?
→ csv

Q: What are the key features of the Wireless Bluetooth Headphones?
→ text

Q: What do customers say about the Air Fryer's ease of cleaning?
→ text

Q: Which product has the best customer reviews and how well is it se
→ hybrid

Q: I want a product for fitness that is highly rated and sells well 
→ hybrid



In [4]:
def chunk_product_file(filename: str, content: str) -> list[dict]:
    """Split product file into chunks at each Header: line."""
    
    chunks = []
    product_id = filename.split("_")[0]
    current_header = "intro"
    current_lines = []
    
    for line in content.split("\n"):
        stripped = line.strip()
        
        # Detect header: line ends with colon and is not a list item
        is_header = (
            stripped.endswith(":")
            and not stripped.startswith("-")
            and not stripped.startswith("*")
            and len(stripped) > 3
            and len(stripped) < 50
        )
        
        if is_header and current_lines:
            text = "\n".join(current_lines).strip()
            if text:
                chunks.append({
                    "product_id": product_id,
                    "filename": filename,
                    "section": current_header,
                    "text": text
                })
            current_lines = []
            current_header = stripped.rstrip(":")
        else:
            current_lines.append(line)
    
    # Save last section
    if current_lines:
        text = "\n".join(current_lines).strip()
        if text:
            chunks.append({
                "product_id": product_id,
                "filename": filename,
                "section": current_header,
                "text": text
            })
    
    return chunks

# Build chunks and embeddings
print("=== Building Text Chunks and Embeddings ===\n")
all_chunks = []

for filename, content in text_files.items():
    chunks = chunk_product_file(filename, content)
    all_chunks.extend(chunks)
    print(f"{filename}: {len(chunks)} chunks → {[c['section'] for c in chunks]}")

print(f"\nTotal chunks: {len(all_chunks)}")

# Generate embeddings
print("\nGenerating embeddings...")
#chunk_texts = [c["text"] for c in all_chunks]
chunk_texts = [f"{c['section']}: {c['text']}" for c in all_chunks]
chunk_embeddings = model.encode(chunk_texts, show_progress_bar=False)

print(f"Embeddings shape: {chunk_embeddings.shape}")
print("✅ Vector store ready")

=== Building Text Chunks and Embeddings ===

TOYS001_product_page: 5 chunks → ['intro', 'PRODUCT DESCRIPTION', 'Set Contents', 'Features', 'CUSTOMER REVIEWS']
SPRT001_product_page: 5 chunks → ['intro', 'PRODUCT DESCRIPTION', 'Key Features', 'Best For', 'CUSTOMER REVIEWS']
OFFC001_product_page: 6 chunks → ['intro', 'PRODUCT DESCRIPTION', 'Key Features', 'Dimensions', 'Certifications', 'CUSTOMER REVIEWS']
BOOK001_product_page: 6 chunks → ['intro', 'PRODUCT DESCRIPTION', 'Book Details', "What You'll Learn", 'Includes', 'CUSTOMER REVIEWS']
BEAU001_product_page: 6 chunks → ['intro', 'PRODUCT DESCRIPTION', 'Key Ingredients', 'Benefits', 'How to Use', 'CUSTOMER REVIEWS']
FOOD001_product_page: 8 chunks → ['intro', 'PRODUCT DESCRIPTION', 'Coffee Profile', 'Sourcing', 'Certifications', 'Package Details', 'Brewing Recommendations', 'CUSTOMER REVIEWS']
HOME003_product_page: 5 chunks → ['intro', 'PRODUCT DESCRIPTION', 'Key Features', 'Technical Specifications', 'CUSTOMER REVIEWS']
PETS001_product_p

In [9]:
def retrieve_from_csv(question: str) -> str:
    """Dynamically answer any analytical question using LLM-generated pandas code."""
    
    # Promt for the  LLM about the dataframe schema
    schema = f"""
You have a pandas DataFrame called `df` with these columns:
- date (datetime): transaction date
- product_id (str): product identifier
- product_name (str): full product name
- category (str): one of {df['category'].unique().tolist()}
- units_sold (int): number of units sold
- unit_price (float): price per unit
- total_revenue (float): total revenue for transaction
- region (str): one of {df['region'].unique().tolist()}
Date range: {df['date'].min().date()} to {df['date'].max().date()}
"""
    
    prompt = f"""{schema}
Write a single Python pandas expression to answer this question:
"{question}"

Rules:
- Return ONLY the pandas code, nothing else
- Must be a single expression that returns a string or number
- Use df as the dataframe name
- Format numbers nicely
- Example: df[df['category']=='Electronics']['total_revenue'].sum()
"""
    
    # LLM generates the pandas code
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    
    pandas_code = response.content[0].text.strip()
    pandas_code = pandas_code.replace("```python", "").replace("```", "").strip()
    print(f"  [CSV] Generated code: {pandas_code}")
    
    # Execute the generated code
    try:
        result = eval(pandas_code)
        return f"Query result: {result}"
    except Exception as e:
        return f"Error executing query: {str(e)}\nCode: {pandas_code}"

# Test on Q1 and Q2
print("=== Testing CSV Retriever ===\n")
q1 = "What was the total revenue for Electronics category in December 2024?"
q2 = "Which region had the highest sales volume?"
q3 = "What is the least popular product in the month of Novemer"
q4 = "How much is the median sales for books different from food in the month of november "

print(f"Q1: {q1}")
print(f"Answer: {retrieve_from_csv(q1)}\n")
print("-" * 50)
print(f"Q2: {q2}")
print(f"Answer: {retrieve_from_csv(q2)}")
print(f"Q3: {q3}")
print(f"Answer: {retrieve_from_csv(q3)}")
print(f"Q4: {q4}")
print(f"Answer: {retrieve_from_csv(q4)}")

=== Testing CSV Retriever ===

Q1: What was the total revenue for Electronics category in December 2024?
  [CSV] Generated code: df[(df['category']=='Electronics') & (df['date'].dt.month==12) & (df['date'].dt.year==2024)]['total_revenue'].sum()
Answer: Query result: 142864.31

--------------------------------------------------
Q2: Which region had the highest sales volume?
  [CSV] Generated code: df.groupby('region')['units_sold'].sum().idxmax()
Answer: Query result: Central
Q3: What is the least popular product in the month of Novemer
  [CSV] Generated code: df[(df['date'] >= '2024-11-01') & (df['date'] < '2024-12-01')].groupby('product_name')['units_sold'].sum().idxmin()
Answer: Query result: Board Game Strategy
Q4: How much is the median sales for books different from food in the month of november 
  [CSV] Generated code: abs(df[(df['category']=='Books') & (df['date'].dt.month==11)]['total_revenue'].median() - df[(df['category']=='Food & Grocery') & (df['date'].dt.month==11)]['total

In [13]:
def expand_query(question: str) -> str:
    """Extract key concepts from question for better semantic matching."""
    
    # Remove question words that confuse embeddings
    noise_words = ["how many", "does the", "what are", "what is", 
                   "can you", "tell me", "i want", "which", "what",
                   "does", "contain", "have", "include", "find"]
    
    q = question.lower()
    for word in noise_words:
        q = q.replace(word, "")
    
    # Clean up extra spaces
    expanded = " ".join(q.split())
    #print(f"  [Text] Expanded: '{question}' → '{expanded}'")
    return expanded

def retrieve_from_text(question: str, top_k: int = 3) -> str:
    """Retrieve relevant product text using semantic search."""
    
    # Expand query
    expanded = expand_query(question)
    
    # Encode expanded query
    question_embedding = model.encode([expanded])
    
    # Compute cosine similarity
    similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]
    
    # Get top k chunks
    top_indices = similarities.argsort()[::-1][:top_k]
    
    context_parts = []
    for idx in top_indices:
        chunk = all_chunks[idx]
        score = similarities[idx]
        context_parts.append(
            f"=== {chunk['product_id']} - {chunk['section']} (score: {score:.3f}) ===\n"
            f"{chunk['text'][:1500]}"
        )
    
    return "\n\n".join(context_parts)

# Test
print("=== Testing Text Retriever ===\n")
q3 = "What are the key features of the Wireless Bluetooth Headphones?"
q4 = "What do customers say about the Air Fryer's ease of cleaning?"


for q in [q3, q4]:
    print(f"Q: {q}")
    context = retrieve_from_text(q)
    print(f"Top chunks:\n{context[:400]}\n")
    print("-" * 50)

=== Testing Text Retriever ===

Q: What are the key features of the Wireless Bluetooth Headphones?
Top chunks:
=== ELEC001 - PRODUCT DESCRIPTION (score: 0.602) ===
Experience crystal-clear audio with our premium Wireless Bluetooth Headphones.
Featuring advanced noise-cancellation technology and 40-hour battery life, these
headphones are perfect for commuting, working from home, or enjoying your favorite
music without interruption.

=== ELEC001 - intro (score: 0.600) ===

--------------------------------------------------
Q: What do customers say about the Air Fryer's ease of cleaning?
Top chunks:
=== HOME003 - CUSTOMER REVIEWS (score: 0.538) ===
----------------------------------------

Review 1 - Patricia W. (Verified Purchase) - 5 stars
"This air fryer has completely changed how I cook! Made the crispiest french fries
I've ever had at home. My kids now prefer my homemade chicken nuggets over fast food.
Easy to clean and the presets work perfectly."

Review 2 - Robert H. (Verified Pur

In [14]:
for chunk in all_chunks:
    if chunk['product_id'] == 'SPRT001':
        print(f"Section: {chunk['section']}")
        print(f"Text preview: {chunk['text'][:200]}")
        print()

question = "fitness products sports outdoors highly rated"
question_embedding = model.encode([question])
similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]

scored = [(all_chunks[i]['product_id'], all_chunks[i]['section'], similarities[i]) 
          for i in range(len(all_chunks))]
scored.sort(key=lambda x: x[2], reverse=True)

for product_id, section, score in scored[:10]:
    print(f"  {product_id} - {section}: {score:.3f}")

Section: intro
Text preview: ========================================
YOGA MAT PREMIUM - PRODUCT PAGE

Product: Yoga Mat Premium
Brand: ZenFlex
Price: $34.99
SKU: SPRT001
Category: Sports 

Section: PRODUCT DESCRIPTION
Text preview: Elevate your yoga practice with the ZenFlex Premium Yoga Mat. Made from eco-friendly
TPE material, this mat provides the perfect balance of cushioning and stability. The
dual-layer construction offers

Section: Key Features
Text preview: - Eco-friendly TPE material, free of PVC, latex, and toxic chemicals
- Extra thick 6mm cushioning for joint protection
- Double-sided non-slip texture
- Lightweight (2.5 lbs) and easy to carry
- Inclu

Section: Best For
Text preview: - Yoga (all styles including hot yoga)
- Pilates
- Floor exercises
- Meditation
- Stretching

Section: CUSTOMER REVIEWS
Text preview: ----------------------------------------

Review 1 - Emily R. (Verified Purchase) - 5 stars
"Finally found the perfect yoga mat! The thickness is just right - my

In [17]:
import re
import json

# Build product ID to name mapping once
id_to_name = {}
for chunk in all_chunks:
    if chunk['section'] == 'intro':
        for line in chunk['text'].split('\n'):
            if 'Product:' in line:
                id_to_name[chunk['product_id']] = line.replace('Product:', '').strip()
                break

def decompose_hybrid_question(question: str) -> dict:
    """Use LLM to decompose hybrid question into text and CSV sub-questions."""
    
    prompt = f"""You are helping a RAG system decompose a hybrid question into two sub-questions.

Available product categories in sales data: {df['category'].unique().tolist()}
Available regions in sales data: {df['region'].unique().tolist()}
Available product files: {list(text_files.keys())}

Question: "{question}"

Decompose into:
1. text_question: what to search in product descriptions and customer reviews.
   Use actual product names or categories from the available product files.
2. csv_question: what to compute from sales data.
   Use exact category names and region names from the available data.
   Always request top 5 products not just top 1.
3. filtered: true if question is about a specific category or region, false if about all products.

Reply ONLY with valid JSON:
{{"text_question": "...", "csv_question": "...", "filtered": true or false}}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    
    text = response.content[0].text.strip()
    text = text.replace("```json", "").replace("```", "").strip()
    return json.loads(text)

def retrieve_hybrid(question: str) -> str:
    """Orchestrate hybrid retrieval by decomposing question and combining results."""
    
    # Step 1 — Decompose question
    sub_questions = decompose_hybrid_question(question)
    csv_question = sub_questions['csv_question']
    is_filtered = sub_questions.get('filtered', False)
    
    # Step 2 — Get CSV results
    csv_context = retrieve_from_csv(csv_question)
    
    if is_filtered:
        # Q6 type — filter reviews to CSV matched products only
        extract_prompt = f"""From this sales data, extract ONLY the product names as a Python list.
Sales Data: {csv_context}
Reply ONLY with a Python list like: ["Product A", "Product B"]"""

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=100,
            messages=[{"role": "user", "content": extract_prompt}]
        )
        
        # Strip markdown code fences before parsing
        product_list_str = response.content[0].text.strip()
        product_list_str = product_list_str.replace("```python", "").replace("```", "").strip()
        try:
            csv_products = json.loads(product_list_str)
        except:
            csv_products = []
        
        # Match CSV products to chunk product IDs
        matching_ids = []
        for chunk in all_chunks:
            if chunk['section'] == 'intro':
                for product in csv_products:
                    if product.lower() in chunk['text'].lower():
                        matching_ids.append(chunk['product_id'])
                        break
        
        # Get reviews for matched products only
        reviews = []
        for chunk in all_chunks:
            if chunk['section'] == 'CUSTOMER REVIEWS' and chunk['product_id'] in matching_ids:
                product_name = id_to_name.get(chunk['product_id'], chunk['product_id'])
                reviews.append(f"=== {product_name} ===\n{chunk['text'][:800]}")
        
        text_context = "\n\n".join(reviews) if reviews else "No matching product reviews found."
    
    else:
        # Q5 type — send ALL reviews, let LLM compare
        all_reviews = []
        for chunk in all_chunks:
            if chunk['section'] == 'CUSTOMER REVIEWS':
                product_name = id_to_name.get(chunk['product_id'], chunk['product_id'])
                all_reviews.append(f"=== {product_name} ===\n{chunk['text'][:800]}")
        
        text_context = "\n\n".join(all_reviews)
    
    # Step 3 — Build combined context
    combined_context = f"""=== PRODUCT REVIEWS ===
{text_context}

=== SALES DATA ===
{csv_context}"""
    
    return combined_context

# Test on Q5 and Q6
print("=== Testing Hybrid Orchestrator ===\n")
q5 = "Which product has the best customer reviews and how well is it selling?"
q6 = "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?"
q7 = "I am looking for a beauty product that is the most expensive but it's sells units are the lowest, please recommend"

for q in [q5, q6, q7]:
    print(f"Q: {q}\n")
    context = retrieve_hybrid(q)
    print(f"Context length: {len(context)}")
    print(f"Context preview:\n{context[:400]}\n")
    print("-" * 50)


=== Testing Hybrid Orchestrator ===

Q: Which product has the best customer reviews and how well is it selling?

  [CSV] Generated code: df.groupby('product_name').agg({'units_sold': 'sum', 'total_revenue': 'sum'}).sort_values('total_revenue', ascending=False).head(5).to_string()
Context length: 8723
Context preview:
=== PRODUCT REVIEWS ===
=== Building Blocks Set 500pc ===
----------------------------------------

Review 1 - Karen M. (Verified Purchase) - 5 stars
"Bought this for my 6-year-old's birthday and he's obsessed! He plays with it for
hours every day. The quality is excellent - fits perfectly with our existing LEGO
collection. Great value compared to name brands."

Review 2 - John D. (Verified Purcha

--------------------------------------------------
Q: I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?

  [CSV] Generated code: df[(df['category'] == 'Sports & Outdoors') & (df['region'] == 'West')].groupby('product_name

In [23]:
def generate_final_answer(question: str, context: str) -> str:
    """Generate final answer using LLM with retrieved context."""
    
    prompt = f"""You are an e-commerce analytics assistant with access to both 
sales data and product information.

Question: {question}

Context:
{context}

Instructions:
- Answer the question directly and clearly
- If sales data is provided, cite specific numbers
- If product reviews are provided, compare ALL product ratings and identify the highest rated
- When comparing reviews, consider both the rating score AND number of reviews 
- If both sources are provided, synthesize insights from both
- Make a clear recommendation if the question asks for one
- Keep the answer concise but complete
- Reference specific products, numbers, and regions where relevant
- DO not ask follow up questions
- DO not say data is missing if it exists in the context
"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.content[0].text.strip()

# Test on all 6 questions
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?"]
    

for q in test_questions:
    print(f"Q: {q}")
    
    route = classify_query(q)
    #print(f"Route: {route}")
    
    if route == "csv":
        context = retrieve_from_csv(q)
    elif route == "text":
        context = retrieve_from_text(q)
    else:
        context = retrieve_hybrid(q)
    
    answer = generate_final_answer(q, context)
    print(f"Answer: {answer}\n")
    print("-" * 50)

    

Q: What was the total revenue for Electronics category in December 2024?
  [CSV] Generated code: df[(df['category']=='Electronics') & (df['date'].dt.month==12) & (df['date'].dt.year==2024)]['total_revenue'].sum()
Answer: # Total Revenue for Electronics Category in December 2024

The total revenue for the Electronics category in December 2024 was **$142,864.31**.

--------------------------------------------------
Q: Which region had the highest sales volume?
  [CSV] Generated code: df.groupby('region')['units_sold'].sum().idxmax()
Answer: Based on the query result provided, **Central region had the highest sales volume**.

--------------------------------------------------
Q: What are the key features of the Wireless Bluetooth Headphones?
Answer: # Key Features of the Wireless Bluetooth Headphones (ELEC001)

The SoundMax Pro Wireless Bluetooth Headphones include the following key features:

1. **Active Noise Cancellation (ANC)** - With transparency mode for switching between immersive 

In [24]:
# Save all results to part2_results.txt
results_path = PROJECT_ROOT / "part2_results.txt"

test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?"
    
]

lines = ["Part 2: Multi-Source RAG Results\n", "=" * 60 + "\n\n"]

for i, q in enumerate(test_questions, 1):
    route = classify_query(q)
    if route == "csv":
        context = retrieve_from_csv(q)
    elif route == "text":
        context = retrieve_from_text(q)
    else:
        context = retrieve_hybrid(q)
    
    answer = generate_final_answer(q, context)
    
    lines.append(f"Q{i}: {q}\n")
    lines.append(f"Answer:\n{answer}\n")
    lines.append("-" * 60 + "\n\n")
    print(f"Q{i} done.")

results_path.write_text("".join(lines))
print(f"\n✅ Results saved to {results_path}")

  [CSV] Generated code: df[(df['category']=='Electronics') & (df['date'].dt.month==12) & (df['date'].dt.year==2024)]['total_revenue'].sum()
Q1 done.
  [CSV] Generated code: df.groupby('region')['units_sold'].sum().idxmax()
Q2 done.
Q3 done.
Q4 done.
  [CSV] Generated code: df.groupby('product_name').agg({'units_sold': 'sum', 'total_revenue': 'sum'}).sort_values('units_sold', ascending=False).head(5).to_string()
Q5 done.

✅ Results saved to /Users/animeliksetyan/Documents/DSAN_AIAgent/spring-2026-a03-Animeliksetyan/part2_results.txt
